# Notebook 01 — Ingestão e Upload Azure
**Projeto:** Steel Plant Analytics — Monitoramento Energético e OEE
**Autor:** Ariel Marquezin
**Stack:** pandas · PyArrow · azure-storage-blob

---

## Objetivo
Ingerir os dados brutos da siderúrgica DAEWOO Steel, validar qualidade,
converter para Parquet e subir para o Azure Blob Storage (camada Bronze).

## Referência Arquitetural — SAP HANA
Em ambiente produtivo (ex: Gerdau), a ingestão viria de views SAP/HANA:
```python
# from hdbcli import dbapi
# conn = dbapi.connect(address='hana-host', port=443,
#                      user='ANALYTICS_USER',
#                      password=os.environ['HANA_PASSWORD'])
# query = '''
#     SELECT TURNO, MAQUINA, CONSUMO_KWH, PRODUCAO_TON, DATA_HORA
#     FROM PRODUCAO_SCHEMA.VW_ENERGIA_PLANTA
#     WHERE DATA_HORA >= ADD_DAYS(CURRENT_DATE, -90)
# '''
# df = pd.read_sql(query, conn)
# conn.close()
# Neste projeto usamos o dataset público DAEWOO Steel como proxy
```

In [ ]:
!pip install pandas pyarrow azure-storage-blob python-dotenv openpyxl -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
from azure.storage.blob import BlobServiceClient

# Paths
RAW_PATH     = '/content/'
BRONZE_PATH  = '/content/drive/MyDrive/steel/bronze/'
os.makedirs(BRONZE_PATH, exist_ok=True)

# Azure — substitua pela sua connection string ou use variavel de ambiente
AZURE_CONN   = os.environ.get('AZURE_STORAGE_CONNECTION_STRING', '')
CONTAINER    = 'steel-bronze'

print('Ambiente configurado OK')
print(f'Azure configurado: {bool(AZURE_CONN)}')

In [ ]:
# ── Leitura do CSV ────────────────────────────────────────────────────────────
# Dataset: Steel Industry Energy Consumption - DAEWOO Steel
# Download: kaggle.com/datasets/csafrit2/steel-industry-energy-consumption
# Arquivo esperado: Steel_industry_data.csv

csv_file = os.path.join(RAW_PATH, 'Steel_industry_data.csv')
df = pd.read_csv(csv_file)

print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Quality Report ────────────────────────────────────────────────────────────
print('=== QUALITY REPORT ===')
print(f'Total de registros    : {len(df):,}')
print(f'Total de colunas      : {len(df.columns)}')
print(f'\nNulos por coluna:')
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
quality = pd.DataFrame({'nulos': nulls, 'pct': nulls_pct})
print(quality[quality['nulos'] > 0].to_string() if quality['nulos'].sum() > 0 else 'Nenhum nulo encontrado')
print(f'\nTipos de dados:')
print(df.dtypes.to_string())

In [ ]:
# ── Tratamento e Enriquecimento ───────────────────────────────────────────────
df_clean = df.copy()

# Padroniza nome das colunas
df_clean.columns = [
    c.strip()
     .lower()
     .replace(' ', '_')
     .replace('(', '')
     .replace(')', '')
     .replace('/', '_')
    for c in df_clean.columns
]
print('Colunas padronizadas:', df_clean.columns.tolist())

# Converte timestamp
date_col = [c for c in df_clean.columns if 'date' in c or 'time' in c][0]
df_clean[date_col] = pd.to_datetime(df_clean[date_col], dayfirst=True, errors='coerce')
df_clean = df_clean.rename(columns={date_col: 'timestamp'})

# Features temporais
df_clean['ano']          = df_clean['timestamp'].dt.year
df_clean['mes']          = df_clean['timestamp'].dt.month
df_clean['dia']          = df_clean['timestamp'].dt.day
df_clean['hora']         = df_clean['timestamp'].dt.hour
df_clean['dia_semana']   = df_clean['timestamp'].dt.dayofweek
df_clean['nome_dia']     = df_clean['timestamp'].dt.day_name()
df_clean['is_fim_semana'] = df_clean['dia_semana'].isin([5, 6]).astype(int)

# Turno (baseado na hora)
def get_turno(hora):
    if 6 <= hora < 14:
        return 'Turno A (06-14h)'
    elif 14 <= hora < 22:
        return 'Turno B (14-22h)'
    else:
        return 'Turno C (22-06h)'

df_clean['turno'] = df_clean['hora'].apply(get_turno)

# Remove duplicatas
antes = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f'Duplicatas removidas: {antes - len(df_clean)}')
print(f'Shape final: {df_clean.shape}')

In [ ]:
# ── Salva Bronze local (Parquet) ──────────────────────────────────────────────
bronze_file = os.path.join(BRONZE_PATH, 'steel_bronze.parquet')
table = pa.Table.from_pandas(df_clean)
pq.write_table(table, bronze_file, compression='snappy')

size_mb = os.path.getsize(bronze_file) / 1024 / 1024
print(f'Bronze salvo: {bronze_file}')
print(f'Tamanho: {size_mb:.2f} MB')
print(f'Registros: {len(df_clean):,}')

In [ ]:
# ── Upload Azure Blob Storage ─────────────────────────────────────────────────
if AZURE_CONN:
    try:
        client = BlobServiceClient.from_connection_string(AZURE_CONN)
        container_client = client.get_container_client(CONTAINER)

        # Cria container se nao existir
        try:
            container_client.create_container()
            print(f'Container {CONTAINER} criado')
        except Exception:
            print(f'Container {CONTAINER} ja existe')

        blob_name = f'bronze/steel_bronze_{datetime.now().strftime("%Y%m%d")}.parquet'
        with open(bronze_file, 'rb') as f:
            container_client.upload_blob(name=blob_name, data=f, overwrite=True)

        print(f'Upload Azure concluido: {blob_name}')
        print(f'Caminho: {CONTAINER}/{blob_name}')
    except Exception as e:
        print(f'Erro no upload Azure: {e}')
else:
    print('AZURE_STORAGE_CONNECTION_STRING nao configurada.')
    print('Configure a variavel de ambiente para habilitar o upload.')
    print('O arquivo Parquet foi salvo localmente no Google Drive.')

print('\nPipeline de ingestao finalizado!')
print(f'  Registros processados : {len(df_clean):,}')
print(f'  Periodo               : {df_clean["timestamp"].min()} a {df_clean["timestamp"].max()}')
print(f'  Turnos identificados  : {df_clean["turno"].nunique()}')